# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Zwiedzanie teatru z przewodnikiem
2. Odpust Emaus 2026
3. Kolorami mnie ratuj
4. Skarby z Łowicza
5. Wystawa XII Konkursu Drzewek Emausowych
6. Cząstki kobiety. Wystawa Barbary Bazger
7. Alive Poster Festival – Post-Competition Exhibition
8. Mosty 2026
9. Jarmark Wielkanocny 2026
10. Kuchnia od kuchni

Czas wykonania: 5.64s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Zwiedzanie teatru z przewodnikiem
2. Odpust Emaus 2026
3. Kolorami mnie ratuj
4. Skarby z Łowicza
5. Wystawa XII Konkursu Drzewek Emausowych
6. Cząstki kobiety. Wystawa Barbary Bazger
7. Alive Poster Festival – Post-Competition Exhibition
8. Mosty 2026
9. Jarmark Wielkanocny 2026
10. Kuchnia od kuchni

Czas wykonania (wątki): 1.69s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [5]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 1.44s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [7]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

NUM_FACTS = 20

# 1. Sekwencyjnie 
start_seq = time.time()
seq_facts = []
for _ in range(NUM_FACTS):
    fact = requests.get(CAT_API_URL).json().get("fact")
    seq_facts.append(fact)
end_seq = time.time()

print("Sekwencyjnie pobrano fakty:")
for f in seq_facts:
    print("-", f)
print(f"Czas sekwencyjny: {end_seq - start_seq:.2f}s\n")

# 2. Wielowątkowo 
def fetch_fact(_):
    return requests.get(CAT_API_URL).json().get("fact")

start_thread = time.time()
thread_facts = []
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    thread_facts = list(executor.map(fetch_fact, range(NUM_FACTS)))
end_thread = time.time()

print("Wielowątkowo pobrano fakty:")
for f in thread_facts:
    print("-", f)
print(f"Czas wielowtkowy: {end_thread - start_thread:.2f}s\n")

# 3. Porównanie 
print(f"Przyspieszenie: { (end_seq - start_seq)/(end_thread - start_thread):.2f}x")

Sekwencyjnie pobrano fakty:
- Some common houseplants poisonous to cats include: English Ivy, iris, mistletoe, philodendron, and yew.
- Unlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.
- All cats have three sets of long hairs that are sensitive to pressure - whiskers, eyebrows,and the hairs between their paw pads.
- The cat has 500 skeletal muscles (humans have 650).
- The technical term for a cat’s hairball is a “bezoar.”
- Cats with long, lean bodies are more likely to be outgoing, and more protective and vocal than those with a stocky build.
- Fossil records from two million years ago show evidence of jaguars.
- Researchers believe the word “tabby” comes from Attabiyah, a neighborhood in Baghdad, Iraq. Tabbies got their name because their striped coats resembled the famous wavy patterns in the silk produced in this city.
- There are approximately 100 breeds of cat.
- A cats field of vision is about 185 degrees.
- A do

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [8]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

NUM_ITEMS = 20  
q = queue.Queue()

def producer():
    for i in range(1, NUM_ITEMS + 1):
        print(f"Producent: dodaje {i}")
        q.put(i)
        time.sleep(0.1) 
    q.put(None)
    q.put(None)

def consumer_even():
    while True:
        item = q.get()
        if item is None:
            q.put(None)  
            break
        if item % 2 == 0:
            print(f"Konsument parzyste: pobrał {item}")
        q.task_done()
        time.sleep(0.05)

def consumer_odd():
    while True:
        item = q.get()
        if item is None:
            q.put(None)  
            break
        if item % 2 == 1:
            print(f"Konsument nieparzyste: pobrał {item}")
        q.task_done()
        time.sleep(0.05)

t_prod = threading.Thread(target=producer)
t_even = threading.Thread(target=consumer_even)
t_odd = threading.Thread(target=consumer_odd)

t_prod.start()
t_even.start()
t_odd.start()

t_prod.join()
t_even.join()
t_odd.join()

Producent: dodaje 1
Producent: dodaje 2
Producent: dodaje 3
Producent: dodaje 4
Producent: dodaje 5
Producent: dodaje 6
Producent: dodaje 7
Producent: dodaje 8
Producent: dodaje 9
Producent: dodaje 10
Producent: dodaje 11
Producent: dodaje 12
Producent: dodaje 13
Producent: dodaje 14
Producent: dodaje 15
Producent: dodaje 16
Producent: dodaje 17
Producent: dodaje 18
Producent: dodaje 19
Producent: dodaje 20


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [10]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def main():
    start_time = time.time()
    numbers = range(1, 10001) 
    with multiprocessing.Pool() as pool:
        results = pool.map(calculate_power_sum, numbers)

    end_time = time.time()
    print(f"Czas wykonania z multiprocessing: {end_time - start_time:.2f} s")

if __name__ == "__main__":
    main()

Czas wykonania z multiprocessing: 0.48 s
